# Task-Level and Block-Level Model Comparison

This notebook:

- loads processed warehouse detailed data
- uses the **last day in the data** as the test set and all earlier days as training
- trains:
  - a **naive mean-time model** by WorkCode
  - a **linear regression** with log response
  - **XGBoost with distance-related features**
  - **XGBoost without distance-related features**
- compares task-level performance
- builds **fixed-length task blocks** on the test set and compares aggregated block-level performance

This notebook **does not** attempt route optimization or task resequencing.


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

CANDIDATE_DIRS = [Path("../data/processed"), Path("../data/process")]
WAREHOUSE = "OE"
WORKCODES = ["10", "20", "30"]
MAX_TIME = 300
BLOCK_SIZE = 10
MIN_ROWS_PER_MODEL = 200
RANDOM_STATE = 42


In [ ]:
def resolve_data_path(warehouse):
    fname = f"{warehouse.lower()}_detailed.parquet"
    for d in CANDIDATE_DIRS:
        p = d / fname
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {fname} in {CANDIDATE_DIRS}")

def tod_bucket(h):
    if 6 <= h < 12:
        return "6-12"
    if 12 <= h < 16:
        return "12-4"
    if 16 <= h < 20:
        return "4-8"
    if 20 <= h < 24:
        return "8-12"
    return "after_midnight"

def engineer_features(df, workcode, max_time=300):
    d = df.copy()
    d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
    d = d.dropna(subset=["Timestamp"]).copy()

    for col in ["Time_Delta_sec", "Travel_Distance", "Distance", "Dist_prev_to_curr", "Weight", "Cube", "Quantity",
                "Aisle", "Bay", "Level", "Prev_Aisle", "Prev_Bay", "Prev_Level"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    d["WorkCode"] = d["WorkCode"].astype(str).str.replace(".0", "", regex=False)
    d = d[d["WorkCode"] == str(workcode)].copy()

    if "Travel_Distance" in d.columns:
        d["Travel_Distance"] = pd.to_numeric(d["Travel_Distance"], errors="coerce")
    elif "Distance" in d.columns:
        d["Travel_Distance"] = pd.to_numeric(d["Distance"], errors="coerce")
    elif "Dist_prev_to_curr" in d.columns:
        d["Travel_Distance"] = pd.to_numeric(d["Dist_prev_to_curr"], errors="coerce")
    else:
        d["Travel_Distance"] = np.nan

    d = d[
        (d["Time_Delta_sec"] > 0) &
        (d["Time_Delta_sec"] < max_time) &
        (d["Weight"] > 0) &
        (d["Cube"] > 0)
    ].copy()

    d = d[d["Travel_Distance"].fillna(0) >= 0].copy()

    d["date"] = d["Timestamp"].dt.date
    d["hour"] = d["Timestamp"].dt.hour.astype(int)
    d["time_of_day"] = d["hour"].apply(tod_bucket)
    d["day_of_week"] = d["Timestamp"].dt.day_name()

    d["Aisle"] = pd.to_numeric(d["Aisle"], errors="coerce").fillna(-1).astype(int)
    top_aisles = d["Aisle"].value_counts().head(10).index.tolist()
    d["Aisle_group"] = d["Aisle"].astype(str).where(d["Aisle"].isin(top_aisles), other="other")
    d["Level_group"] = pd.to_numeric(d["Level"], errors="coerce").fillna(-1).astype(int).astype(str)

    if "Prev_Aisle" in d.columns:
        d["same_aisle"] = np.where(d["Aisle"] == d["Prev_Aisle"], "yes", "no")
    else:
        d["same_aisle"] = "unknown"

    if {"LocationID", "Prev_LocationID"}.issubset(d.columns):
        d["same_location"] = np.where(d["LocationID"] == d["Prev_LocationID"], "yes", "no")
    else:
        d["same_location"] = "unknown"

    if "Prev_Level" in d.columns:
        d["diff_level"] = np.where(d["Level"] != d["Prev_Level"], "yes", "no")
    else:
        d["diff_level"] = "unknown"

    if "UnitOfMeasure" in d.columns:
        keep = {"EA", "BX", "PK", "CA", "CS"}
        u = d["UnitOfMeasure"].astype(str).str.upper()
        d["UOM_group"] = u.where(u.isin(keep), other="other")
    else:
        d["UOM_group"] = "other"

    d["log_weight"] = np.log(d["Weight"])
    d["log_cube"] = np.log(d["Cube"])
    d["log_quantity"] = np.log1p(pd.to_numeric(d["Quantity"], errors="coerce").fillna(0).clip(lower=0))
    d["log_travel_distance"] = np.log1p(d["Travel_Distance"])

    base_num = ["Weight", "Cube", "Quantity", "log_weight", "log_cube", "log_quantity"]
    dist_num = ["Travel_Distance", "log_travel_distance"]
    cat_cols = ["Aisle_group", "Level_group", "time_of_day", "same_aisle", "same_location", "diff_level", "UOM_group", "day_of_week"]

    features_with_distance = base_num + dist_num + cat_cols
    features_without_distance = base_num + cat_cols

    return d, features_with_distance, features_without_distance, cat_cols

def split_last_day(df):
    all_days = sorted(df["date"].dropna().unique())
    if len(all_days) < 2:
        raise ValueError("Need at least 2 distinct days to split on the last day.")
    test_day = all_days[-1]
    train_df = df[df["date"] < test_day].copy()
    test_df = df[df["date"] == test_day].copy()
    return train_df, test_df, test_day

def make_X(train_df, test_df, features, cat_cols):
    X_train = pd.get_dummies(train_df[features], columns=cat_cols, drop_first=True)
    X_test = pd.get_dummies(test_df[features], columns=cat_cols, drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    return X_train, X_test

def eval_predictions(y_true, pred):
    return {
        "r2": r2_score(y_true, pred),
        "mae": mean_absolute_error(y_true, pred),
        "rmse": np.sqrt(mean_squared_error(y_true, pred)),
    }

def make_test_blocks(test_df, block_size=10):
    d = test_df.sort_values(["UserID", "Timestamp"]).copy()
    blocks = []
    block_rows = []

    for (uid, day), g in d.groupby(["UserID", "date"], sort=False):
        g = g.sort_values("Timestamp").reset_index().rename(columns={"index": "orig_index"}).copy()
        n = len(g)
        for start in range(0, n, block_size):
            chunk = g.iloc[start:start + block_size].copy()
            if len(chunk) < block_size:
                continue
            if chunk["WorkCode"].nunique() != 1:
                continue

            block_id = f"{uid}_{day}_{start//block_size}"
            chunk["BlockID"] = block_id
            block_rows.append(chunk)

            blocks.append({
                "BlockID": block_id,
                "UserID": uid,
                "date": day,
                "WorkCode": chunk["WorkCode"].iloc[0],
                "n_tasks": len(chunk),
                "actual_time": chunk["Time_Delta_sec"].sum(),
                "start_ts": chunk["Timestamp"].min(),
                "end_ts": chunk["Timestamp"].max(),
            })

    block_df = pd.DataFrame(blocks)
    block_rows_df = pd.concat(block_rows, ignore_index=True) if block_rows else pd.DataFrame()
    return block_df, block_rows_df


In [ ]:
data_path = resolve_data_path(WAREHOUSE)
raw_df = pd.read_parquet(data_path)

all_task_results = []
all_block_results = []
all_block_detail = []

for wc in WORKCODES:
    df_wc, features_wd, features_nod, cat_cols = engineer_features(raw_df, workcode=wc, max_time=MAX_TIME)

    if len(df_wc) < MIN_ROWS_PER_MODEL:
        print(f"Skipping WC {wc}: too few rows ({len(df_wc)})")
        continue

    train_df, test_df, test_day = split_last_day(df_wc)

    if len(train_df) < MIN_ROWS_PER_MODEL or len(test_df) == 0:
        print(f"Skipping WC {wc}: insufficient split. train={len(train_df)}, test={len(test_df)}")
        continue

    print(f"WC {wc} | train={len(train_df)} | test={len(test_df)} | last_day={test_day}")

    y_train = train_df["Time_Delta_sec"].astype(float)
    y_test = test_df["Time_Delta_sec"].astype(float)

    # Naive mean
    t0 = time.perf_counter()
    naive_mean = y_train.mean()
    pred_naive = np.repeat(naive_mean, len(y_test))
    naive_runtime = time.perf_counter() - t0
    m_naive = eval_predictions(y_test, pred_naive)
    all_task_results.append({
        "Warehouse": WAREHOUSE, "WorkCode": wc, "Model": "Naive mean",
        "n_train_rows": len(train_df), "n_test_rows": len(test_df), "runtime_sec": naive_runtime, **m_naive
    })

    # Linear regression with log response
    X_train_lr, X_test_lr = make_X(train_df, test_df, features_wd, cat_cols)
    t0 = time.perf_counter()
    lr = LinearRegression()
    lr.fit(X_train_lr, np.log(y_train))
    pred_lr = np.exp(lr.predict(X_test_lr))
    lr_runtime = time.perf_counter() - t0
    m_lr = eval_predictions(y_test, pred_lr)
    all_task_results.append({
        "Warehouse": WAREHOUSE, "WorkCode": wc, "Model": "Linear regression (log y)",
        "n_train_rows": len(train_df), "n_test_rows": len(test_df), "runtime_sec": lr_runtime, **m_lr
    })

    # XGBoost with distance
    X_train_xgb_d, X_test_xgb_d = make_X(train_df, test_df, features_wd, cat_cols)
    t0 = time.perf_counter()
    xgb_d = XGBRegressor(
        n_estimators=1200, learning_rate=0.03, max_depth=6, min_child_weight=3,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0,
        objective="reg:squarederror", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    xgb_d.fit(X_train_xgb_d, y_train)
    pred_xgb_d = xgb_d.predict(X_test_xgb_d)
    xgb_d_runtime = time.perf_counter() - t0
    m_xgb_d = eval_predictions(y_test, pred_xgb_d)
    all_task_results.append({
        "Warehouse": WAREHOUSE, "WorkCode": wc, "Model": "XGBoost (with distance)",
        "n_train_rows": len(train_df), "n_test_rows": len(test_df), "runtime_sec": xgb_d_runtime, **m_xgb_d
    })

    # XGBoost without distance
    X_train_xgb_n, X_test_xgb_n = make_X(train_df, test_df, features_nod, cat_cols)
    t0 = time.perf_counter()
    xgb_n = XGBRegressor(
        n_estimators=1200, learning_rate=0.03, max_depth=6, min_child_weight=3,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0,
        objective="reg:squarederror", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    xgb_n.fit(X_train_xgb_n, y_train)
    pred_xgb_n = xgb_n.predict(X_test_xgb_n)
    xgb_n_runtime = time.perf_counter() - t0
    m_xgb_n = eval_predictions(y_test, pred_xgb_n)
    all_task_results.append({
        "Warehouse": WAREHOUSE, "WorkCode": wc, "Model": "XGBoost (without distance)",
        "n_train_rows": len(train_df), "n_test_rows": len(test_df), "runtime_sec": xgb_n_runtime, **m_xgb_n
    })

    # block-level evaluation
    block_df, block_rows_df = make_test_blocks(test_df, block_size=BLOCK_SIZE)
    if len(block_df) == 0:
        print(f"No valid blocks for WC {wc}")
        continue

    temp = test_df.copy().reset_index().rename(columns={"index": "orig_index"})
    temp["pred_naive"] = pred_naive
    temp["pred_lr"] = pred_lr
    temp["pred_xgb_with_dist"] = pred_xgb_d
    temp["pred_xgb_no_dist"] = pred_xgb_n

    block_rows_df = block_rows_df.merge(
        temp[["orig_index", "pred_naive", "pred_lr", "pred_xgb_with_dist", "pred_xgb_no_dist"]],
        on="orig_index",
        how="left"
    )

    block_pred = (
        block_rows_df.groupby("BlockID")
        .agg(
            actual_time=("Time_Delta_sec", "sum"),
            pred_naive=("pred_naive", "sum"),
            pred_lr=("pred_lr", "sum"),
            pred_xgb_with_dist=("pred_xgb_with_dist", "sum"),
            pred_xgb_no_dist=("pred_xgb_no_dist", "sum"),
            WorkCode=("WorkCode", "first"),
            UserID=("UserID", "first"),
            date=("date", "first"),
            n_tasks=("Time_Delta_sec", "size"),
        )
        .reset_index()
    )

    for model_name, col in [
        ("Naive mean", "pred_naive"),
        ("Linear regression (log y)", "pred_lr"),
        ("XGBoost (with distance)", "pred_xgb_with_dist"),
        ("XGBoost (without distance)", "pred_xgb_no_dist"),
    ]:
        metrics = eval_predictions(block_pred["actual_time"], block_pred[col])
        all_block_results.append({
            "Warehouse": WAREHOUSE, "WorkCode": wc, "Model": model_name,
            "n_blocks": len(block_pred), **metrics
        })

    block_pred["Warehouse"] = WAREHOUSE
    all_block_detail.append(block_pred)

task_results_df = pd.DataFrame(all_task_results)
block_results_df = pd.DataFrame(all_block_results)
block_detail_df = pd.concat(all_block_detail, ignore_index=True) if all_block_detail else pd.DataFrame()


In [ ]:
task_results_clean = task_results_df.copy()
for c in ["r2", "mae", "rmse", "runtime_sec"]:
    if c in task_results_clean.columns:
        task_results_clean[c] = task_results_clean[c].round(4)

task_results_clean = task_results_clean.sort_values(["WorkCode", "mae", "r2"], ascending=[True, True, False]).reset_index(drop=True)
display(task_results_clean)


In [ ]:
block_results_clean = block_results_df.copy()
for c in ["r2", "mae", "rmse"]:
    if c in block_results_clean.columns:
        block_results_clean[c] = block_results_clean[c].round(4)

block_results_clean = block_results_clean.sort_values(["WorkCode", "mae", "r2"], ascending=[True, True, False]).reset_index(drop=True)
display(block_results_clean)


In [ ]:
display(block_detail_df.head(20))


In [ ]:
plt.figure(figsize=(10, 5))
plot_df = task_results_clean.copy()
plot_df["label"] = plot_df["WorkCode"].astype(str) + " | " + plot_df["Model"]
plt.bar(plot_df["label"], plot_df["mae"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Task-level MAE (seconds)")
plt.title(f"{WAREHOUSE}: Task-level model comparison")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plot_df2 = block_results_clean.copy()
plot_df2["label"] = plot_df2["WorkCode"].astype(str) + " | " + plot_df2["Model"]
plt.bar(plot_df2["label"], plot_df2["mae"])
plt.xticks(rotation=45, ha="right")
plt.ylabel(f"Block-level MAE (seconds), block size={BLOCK_SIZE}")
plt.title(f"{WAREHOUSE}: Block-level model comparison")
plt.tight_layout()
plt.show()


## Notes

- The **last calendar day** in the filtered data for each WorkCode is used as the test set.
- This notebook does **not** attempt route optimization or infer true Unit of Work / Assignment groupings.
- Block-level evaluation uses **fixed-length chunks of 10 tasks** within each user-day, keeping only chunks with a **single WorkCode**.
- Aggregated block predictions are created by summing task-level predictions.
